# Depth 1.1 — PC stability across seeds + PC1 confound test

**Two open questions left by Depth 1 (seed 0)**:

1. **The "important" PC moved**. The pre-existing note said PC2 carried the cross-modal
   signal (R²=0.26). Depth 1 at seed 0 found PC2 R² ≈ 0 and PC3 R² = 0.26 instead.
   PC indices are seed-dependent (and signs are arbitrary), so a single-seed table cannot
   tell us whether there is a *physical mode* that consistently has high FC-R² + MZ
   separation, or whether the "important PC" jumps around unpredictably across seeds.

2. **PC1 might be sex + brain volume in disguise**. PC1 had FC->R²=0.59, AUC_MZ=0.82,
   AUC_sibling=0.49. A demographic-shared component (sex one-hot + 16 FreeSurfer volumes)
   would look exactly like this: MZ twins are sex-matched and have very similar BV;
   DZ/siblings less so, unrelated subjects not at all. Until we residualize PC1 on
   sex+BV and check what's left, the "FC predicts a heritable structural mode" reading
   is not falsifiable.

This notebook resolves both. **Gating logic**: Section A first. If no PC has stable
high FC-R² + MZ separation across seeds, the mechanism story is dead regardless of
Section B's result.

## Plan

| Section | Question | Cost |
|---|---|---|
| A | 10-seed PC stability: align top-10 SC PCs across seeds by loading cosine similarity; for each stable mode, report median FC-R² + AUCs | ~10 min |
| B | PC1 confound test: regress PC1 scores ~ sex + brain volume at seed 0; check whether the residualized PC1 is still FC-predictable | ~10 s |
| C | Synthesis: combine the two — decide which of {real low-dim mechanism, MZ-only artifact, PC1=demographic-confound} the data supports | instant |

Output: `../model_overviews/results/local_results/further_exploration/depth1.1_pc_stability_and_confounds/`


In [ ]:
# ============= SETUP =============
import sys
from pathlib import Path
for _cand in [Path.cwd(),
              Path.cwd() / "notebooks-FC_to_SC-experimental" / "further_exploration",
              Path.cwd().parent,
              Path.cwd().parent.parent]:
    if (_cand / "_setup.py").exists():
        sys.path.insert(0, str(_cand))
        break
else:
    raise RuntimeError(f"_setup.py not found from cwd={Path.cwd()}")
from _setup import *
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

K_PCA_TOP = 10              # top PCs to track
K_PCA_FC  = 256             # FC PCA dimensionality for FC->PC R² regression
N_SEEDS   = 10
SEEDS     = list(range(N_SEEDS))
out_dir   = results_dir("depth1.1_pc_stability_and_confounds")
print(f"out_dir = {out_dir}")


---

## Section A — 10-seed PC stability via loading alignment

For each seed, refit PCA(SC_train, K=10) and PCA(FC_train, K=256). For each of the
top-10 SC PCs, compute (i) FC→PC R² via BayesianRidge on the 256-D FC latent, and
(ii) AUC(MZ/DZ/sibling vs unrelated_matched) on the per-subject PC scores.

**Alignment**: PC indices are not comparable across seeds, and signs are arbitrary
(sklearn PCA flips signs unpredictably). To compare a "physical" mode across seeds,
use seed 0 as anchor and for each of its top-10 PCs, find the best match in every
other seed by **max |cosine similarity| of loading vectors**. The matched PC carries
the same physical mode (up to sign), regardless of its index.

For each anchor PC, then report:
  - median |cosine sim| across the 9 matches (stability score, in [0, 1])
  - median FC→R² across 10 seeds
  - median AUCs across 10 seeds
  - the matched PC index in each seed (so you can see how much the index moved)

**Decision rule**:
  - A PC is *stable* if median |cosine sim| ≥ 0.85 (low end of "same physical mode").
  - A *stable AND FC-predictable AND heritable* PC is the mechanism candidate.


In [ ]:
# ===== Section A: per-seed PCA + alignment to seed-0 loadings =====
# Per-seed cache: loadings (top-10 SC PCs), FC->R² (per PC), AUCs (per PC), explained_var_ratio.
per_seed = {}

for seed in SEEDS:
    print(f"--- seed {seed} ---", flush=True)
    sp = load_seed_split(seed=seed)
    SC_tr_s, SC_te_s = sp["SC_train"], sp["SC_test"]
    FC_tr_s, FC_te_s = sp["FC_train"], sp["FC_test"]
    base_s, test_idx_s = sp["base"], sp["test_idx"]

    pca_sc_s = PCA(n_components=K_PCA_TOP, random_state=0).fit(SC_tr_s)
    pca_fc_s = PCA(n_components=K_PCA_FC,  random_state=0).fit(FC_tr_s)
    Z_FC_tr  = pca_fc_s.transform(FC_tr_s)
    Z_FC_te  = pca_fc_s.transform(FC_te_s)
    sc_scores_tr = pca_sc_s.transform(SC_tr_s)
    sc_scores_te = pca_sc_s.transform(SC_te_s)

    # Per-PC FC->R²
    fc_r2 = np.zeros(K_PCA_TOP, dtype=np.float64)
    for k in range(K_PCA_TOP):
        m = BayesianRidge(max_iter=300).fit(Z_FC_tr, sc_scores_tr[:, k])
        pred = m.predict(Z_FC_te)
        fc_r2[k] = r2_score(sc_scores_te[:, k], pred)

    # Per-PC AUCs
    rng = np.random.default_rng(42 + seed)  # vary pair sampling per seed
    pairs = pair_indices_by_relation(base_s.metadata_df, test_idx_s, rng, age_tol_yrs=3.0)
    auc_mz = np.full(K_PCA_TOP, np.nan)
    auc_dz = np.full(K_PCA_TOP, np.nan)
    auc_sb = np.full(K_PCA_TOP, np.nan)
    for k in range(K_PCA_TOP):
        score_vec = sc_scores_te[:, k]
        diff = np.abs(score_vec[:, None] - score_vec[None, :])
        sims_by_rel = extract_pair_sims(-diff, pairs)
        a = auc_vs_unrelated(sims_by_rel)
        auc_mz[k] = a.get("MZ", np.nan)
        auc_dz[k] = a.get("DZ", np.nan)
        auc_sb[k] = a.get("sibling", np.nan)

    per_seed[seed] = {
        "loadings":   pca_sc_s.components_.copy(),
        "expl_var":   pca_sc_s.explained_variance_ratio_.copy(),
        "fc_r2":      fc_r2,
        "auc_mz":     auc_mz,
        "auc_dz":     auc_dz,
        "auc_sb":     auc_sb,
        "pair_counts": {k: len(v) for k, v in pairs.items()},
    }
    print(f"  pair counts: {per_seed[seed]['pair_counts']}")
    print(f"  FC->R² (PC1..PC10): {[f'{x:.3f}' for x in fc_r2]}")

print("\nDone refitting 10 seeds.")


In [ ]:
# ===== Section A.2: align top-10 PCs to seed-0 by loading cosine similarity =====
anchor_loadings = per_seed[0]["loadings"]   # shape (K, n_edges)

# For each anchor PC k, find best-matching PC in each other seed (max |cosine sim|).
def cosine_sim_matrix(A, B):
    A = A / np.linalg.norm(A, axis=1, keepdims=True).clip(1e-12)
    B = B / np.linalg.norm(B, axis=1, keepdims=True).clip(1e-12)
    return A @ B.T

aligned_rows = []
for k in range(K_PCA_TOP):
    # Stack stats across seeds for the matched PC.
    match_indices = {0: (k, 1.0)}   # anchor seed: trivially itself, |cos|=1
    fc_r2_seeds = [per_seed[0]["fc_r2"][k]]
    mz_seeds    = [per_seed[0]["auc_mz"][k]]
    dz_seeds    = [per_seed[0]["auc_dz"][k]]
    sb_seeds    = [per_seed[0]["auc_sb"][k]]
    var_seeds   = [per_seed[0]["expl_var"][k]]
    cos_seeds   = []   # |cosine sim| of best match (excluding anchor)
    for s in SEEDS[1:]:
        sim_row = cosine_sim_matrix(anchor_loadings[k:k+1], per_seed[s]["loadings"])[0]
        j = int(np.argmax(np.abs(sim_row)))
        cos_seeds.append(float(np.abs(sim_row[j])))
        match_indices[s] = (j, float(sim_row[j]))   # signed sim too
        fc_r2_seeds.append(per_seed[s]["fc_r2"][j])
        mz_seeds.append(per_seed[s]["auc_mz"][j])
        dz_seeds.append(per_seed[s]["auc_dz"][j])
        sb_seeds.append(per_seed[s]["auc_sb"][j])
        var_seeds.append(per_seed[s]["expl_var"][j])

    aligned_rows.append({
        "anchor_pc":           k + 1,
        "median_abs_cos":      float(np.median(cos_seeds)),
        "min_abs_cos":         float(np.min(cos_seeds)),
        "median_expl_var":     float(np.median(var_seeds)),
        "median_FC_to_PC_R2":  float(np.median(fc_r2_seeds)),
        "median_AUC_MZ":       float(np.nanmedian(mz_seeds)),
        "median_AUC_DZ":       float(np.nanmedian(dz_seeds)),
        "median_AUC_sibling":  float(np.nanmedian(sb_seeds)),
        "matched_indices":     str({s: match_indices[s][0] + 1 for s in SEEDS}),  # 1-indexed
    })

aligned_df = pd.DataFrame(aligned_rows)
print("Per-anchor-PC stats (medians across 10 seeds, alignment by loading cosine):")
print(aligned_df.to_string(index=False, float_format=lambda x: f"{x:7.4f}"))

aligned_df.to_csv(out_dir / "stability_aligned_to_seed0.csv", index=False)
print(f"\nSaved -> {out_dir / 'stability_aligned_to_seed0.csv'}")

# Decision: which anchor PCs are stable?
STABLE_COS_THRESH = 0.85
stable = aligned_df[aligned_df["median_abs_cos"] >= STABLE_COS_THRESH]
print(f"\nStable anchor PCs (median |cos| >= {STABLE_COS_THRESH}):")
if len(stable) == 0:
    print("  NONE. The single-seed PC story is an artifact — the cross-modal signal is not")
    print("  spectrally concentrated.")
else:
    print(stable[["anchor_pc", "median_abs_cos", "median_FC_to_PC_R2",
                  "median_AUC_MZ", "median_AUC_DZ", "median_AUC_sibling"]]
          .to_string(index=False, float_format=lambda x: f"{x:7.4f}"))

# Spearman across stable PCs.
if len(aligned_df) >= 3:
    for col in ("median_AUC_MZ", "median_AUC_DZ", "median_AUC_sibling"):
        v = aligned_df[col].values
        r = aligned_df["median_FC_to_PC_R2"].values
        mask = ~(np.isnan(v) | np.isnan(r))
        if mask.sum() >= 3:
            rho, p = spearmanr(v[mask], r[mask])
            print(f"  Spearman({col}, median_FC_to_PC_R2)  rho={rho:+.3f}  p={p:.3f}")


---

## Section B — PC1 confound test

PC1 at seed 0 looked like a "heritable structural mode FC can predict":
FC→R²=0.59, AUC_MZ=0.82. But MZ twins are sex-matched AND share brain volume tightly,
so a demographic component would have an MZ-only family signature like that just from
being sex+BV.

**Test**: at seed 0, regress test-set PC1 scores ~ [sex_onehot, FreeSurfer 16 volumes]
via OLS. Then:
  - If OLS R² ≈ 1.0 → PC1 IS sex+BV; the "FC predicts heritable structural mode"
    reading collapses to "FC predicts demographic confounds."
  - If OLS R² ≈ 0 → PC1 is independent of sex+BV; the structural mode is real.
  - If OLS R² intermediate → PC1 is partly demographic. Residualize and test
    whether the *residualized* PC1 is still FC-predictable.

This is the falsifier the seed-0 reading needs before anyone writes "FC predicts a
heritable structural backbone."


In [ ]:
# ===== Section B: PC1 = sex + brain volume? =====
# Seed 0 (matches Depth 1).
sp0 = load_seed_split(seed=0)
SC_tr0, SC_te0 = sp0["SC_train"], sp0["SC_test"]
FC_tr0, FC_te0 = sp0["FC_train"], sp0["FC_test"]
bv_tr0, bv_te0 = sp0["bv_train"], sp0["bv_test"]
demo_tr0, demo_te0 = sp0["demo_train"], sp0["demo_test"]
base0, test_idx0 = sp0["base"], sp0["test_idx"]

pca_sc0 = PCA(n_components=10, random_state=0).fit(SC_tr0)
SC_pc_tr = pca_sc0.transform(SC_tr0)
SC_pc_te = pca_sc0.transform(SC_te0)

# Sex one-hot (already in demo_tr0 as 2 cols after age_z), but to be explicit:
sex_tr = base0.sex_oh[sp0["train_idx"]]      # one-hot, 2 cols
sex_te = base0.sex_oh[sp0["test_idx"]]
# Sex + brain volume features.
X_confound_tr = np.concatenate([sex_tr, bv_tr0], axis=1)
X_confound_te = np.concatenate([sex_te, bv_te0], axis=1)
print(f"Confound design: sex({sex_tr.shape[1]}) + bv({bv_tr0.shape[1]}) = "
      f"{X_confound_tr.shape[1]}-dim")

# Fit on TRAIN, eval R² on TEST for each of top 10 PCs.
confound_rows = []
for k in range(10):
    ols = LinearRegression().fit(X_confound_tr, SC_pc_tr[:, k])
    pred_te = ols.predict(X_confound_te)
    r2_te = r2_score(SC_pc_te[:, k], pred_te)
    pred_tr = ols.predict(X_confound_tr)
    r2_tr = r2_score(SC_pc_tr[:, k], pred_tr)
    confound_rows.append({
        "pc": k + 1,
        "confound_R2_train": r2_tr,
        "confound_R2_test":  r2_te,
    })

confound_df = pd.DataFrame(confound_rows)
print("\nOLS [sex || bv] -> SC_PC_k  (R²):")
print(confound_df.to_string(index=False, float_format=lambda x: f"{x:7.4f}"))
confound_df.to_csv(out_dir / "pc_confound_r2.csv", index=False)
print(f"\nSaved -> {out_dir / 'pc_confound_r2.csv'}")

# Residualize PC1 on [sex || bv] and re-test FC -> residualized PC1.
print("\n--- Residualize PC1 on sex+bv, retest FC predictability ---")
ols_pc1 = LinearRegression().fit(X_confound_tr, SC_pc_tr[:, 0])
pc1_resid_tr = SC_pc_tr[:, 0] - ols_pc1.predict(X_confound_tr)
pc1_resid_te = SC_pc_te[:, 0] - ols_pc1.predict(X_confound_te)

pca_fc0 = PCA(n_components=K_PCA_FC, random_state=0).fit(FC_tr0)
Z_FC_tr0 = pca_fc0.transform(FC_tr0)
Z_FC_te0 = pca_fc0.transform(FC_te0)

br_raw   = BayesianRidge(max_iter=300).fit(Z_FC_tr0, SC_pc_tr[:, 0])
r2_raw   = r2_score(SC_pc_te[:, 0], br_raw.predict(Z_FC_te0))
br_resid = BayesianRidge(max_iter=300).fit(Z_FC_tr0, pc1_resid_tr)
r2_resid = r2_score(pc1_resid_te,    br_resid.predict(Z_FC_te0))

print(f"  FC -> PC1 (raw)         R² = {r2_raw:.4f}")
print(f"  FC -> PC1 (resid s+bv)  R² = {r2_resid:.4f}")
print(f"  drop = {r2_raw - r2_resid:+.4f}")

# Also recompute family AUCs on residualized PC1 to see if the MZ signal survives.
rng = np.random.default_rng(42)
pairs0 = pair_indices_by_relation(base0.metadata_df, test_idx0, rng, age_tol_yrs=3.0)

def per_pc_aucs(vec, pairs):
    diff = np.abs(vec[:, None] - vec[None, :])
    sims_by_rel = extract_pair_sims(-diff, pairs)
    return auc_vs_unrelated(sims_by_rel)

a_raw   = per_pc_aucs(SC_pc_te[:, 0], pairs0)
a_resid = per_pc_aucs(pc1_resid_te,   pairs0)
print(f"\n  PC1 AUCs (raw):       {a_raw}")
print(f"  PC1 AUCs (residualized): {a_resid}")


---

## Section C — Synthesis: what does the data actually support?

Combines Section A (PC stability) and Section B (PC1 confound) into a single verdict.

Decision tree (executed below):

```
IF Section A finds no stable PC with high FC-R² + MZ separation:
  -> "Cross-modal signal is distributed, not spectrally concentrated." Mechanism = REJECTED.

ELIF Section B finds PC1 confound R² > 0.7 AND residualized PC1 FC-R² < 0.10:
  -> "PC1 was sex+BV in disguise; FC predicts a demographic confound."
     The 'heritable structural mode' reading was an artifact.

ELIF stable PCs exist AND PC1 confound R² < 0.4:
  -> "FC predicts a stable structural mode that is heritable beyond demographics."
     Mechanism = CONFIRMED.

ELSE:
  -> "Mechanism is partial." Report the stable+heritable PCs but flag the demographic
     overlap. Cross-modal signal exists but is entangled with anatomical priors.
```


In [ ]:
# ===== Section C: synthesis verdict =====
# Pull the numbers from Sections A and B.
stable_mech_PCs = aligned_df[
    (aligned_df["median_abs_cos"] >= 0.85) &
    (aligned_df["median_FC_to_PC_R2"] >= 0.15) &
    (aligned_df["median_AUC_MZ"] >= 0.60)
]
print("Stable + FC-predictable + MZ-separating PCs:")
if len(stable_mech_PCs) == 0:
    print("  NONE.")
else:
    print(stable_mech_PCs[["anchor_pc", "median_abs_cos", "median_FC_to_PC_R2",
                            "median_AUC_MZ", "median_AUC_DZ", "median_AUC_sibling"]]
          .to_string(index=False, float_format=lambda x: f"{x:7.4f}"))

pc1_conf_r2  = confound_df.query("pc == 1").iloc[0]["confound_R2_test"]
pc1_fc_drop  = r2_raw - r2_resid

print(f"\nPC1 confound (test) R² (sex+bv -> PC1)   = {pc1_conf_r2:.4f}")
print(f"PC1 FC predictability  raw vs residualized = {r2_raw:.4f} vs {r2_resid:.4f}  (drop {pc1_fc_drop:+.4f})")

print("\n" + "=" * 70)
print("AUTOMATED VERDICT")
print("=" * 70)

if len(stable_mech_PCs) == 0:
    print("  -> NO stable PC has high FC-R² + MZ AUC across 10 seeds.")
    print("     The cross-modal signal is DISTRIBUTED, not spectrally concentrated.")
    print("     Mechanism story REJECTED. The asymmetry is real but does not factor")
    print("     into a low-dim heritable backbone.")
elif pc1_conf_r2 > 0.70 and r2_resid < 0.10 and 1 in stable_mech_PCs["anchor_pc"].values:
    print("  -> PC1 IS largely sex+BV (confound R² > 0.70). After residualization,")
    print(f"     FC predictability collapses ({r2_raw:.3f} -> {r2_resid:.3f}).")
    print("     If PC1 was the only 'mechanism' candidate, the heritable-mode reading")
    print("     was a demographic confound. Check the other stable PCs separately.")
elif len(stable_mech_PCs) > 0 and pc1_conf_r2 < 0.40:
    print("  -> Stable PCs with high FC-R² + MZ separation exist, and PC1 is NOT")
    print("     a strong sex+BV confound. Mechanism CONFIRMED: FC predicts a")
    print("     stable structural mode heritable beyond demographics.")
else:
    print("  -> Mechanism PARTIAL. Stable+heritable+FC-predictable PCs exist but")
    print("     overlap with demographic priors. Cross-modal signal is real but")
    print("     entangled with anatomy. Report both numbers, do not over-claim.")

print("\nDetail: matched-index drift across seeds (column = seed, value = matched PC):")
print(aligned_df[["anchor_pc", "matched_indices"]].to_string(index=False))
